This notebook implements the article "Power particles: an incompressible fluid solver based on power diagrams" by de Goes et al.

In contrast with the fluid model of Gallouet & Mérigot (implemented in fluid-2d.ipynb), it projects the velocity onto the set of incompressible field through the resolution of a linear system.

Throughout this notebook, we will plot animation of the fluid. 
Since it's hard to deal with animations in Jupyter Notebooks, we prefer to plot mosaics.
There is also a `VIDEO` mode, saving all plots in a folder. You can then use `ffmpeg` to create the video with:
```sh
ffmpeg -i "%04d.png" -c:v libx264 -pix_fmt yuv420p  out.mp4
```

In [1]:
import os
import geogram as geo
import numpy as np
from tqdm import trange, tqdm

import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.collections
from matplotlib.collections import LineCollection

import scipy
import geogram as geo

# Note that we freeze the seed to ensure everyone will have the same results.
np.random.seed(0)

In [2]:
def centroids(voronoi):
    _, inverse = np.unique(voronoi.tseed, return_inverse=True, return_counts=False)

    # Compute cell decomposition centroids
    triangles_areas    = triangle_area(voronoi.q, voronoi.t)
    triangle_centroids = np.sum(voronoi.q[voronoi.t], axis=1) / 3.
    weighted_centroids = triangles_areas[:, np.newaxis] * triangle_centroids

    areas         = np.zeros(len(voronoi.seeds))
    centroids_sum = np.zeros((len(voronoi.seeds), 2))

    np.add.at(centroids_sum, inverse, weighted_centroids)
    np.add.at(areas, inverse, triangles_areas)

    return centroids_sum / areas[:, np.newaxis]


def facets(voronoi):
    NO_INDEX = -1  # Special value for invalid indices (edge on border)

    # Compute one entry per triangle half-edge (3*nt entries) with:
    # I is the seed associated with the triangle
    # J is the seed on the other side of the triangle's edge (NO_INDEX on border)
    # V1 and V2 are the two vertices of the triangle
    i = voronoi.tseed

    tadj = voronoi.tadj
    tadj = tadj[:, [1, 2, 0]] # <- permute columns to match std convention for triangulations:
                              #   different in geogram meshes because they also support n-sided polygons
                                      
    j = tadj.T.flatten().astype(np.int32)
    j = np.where(j != NO_INDEX, i[j], NO_INDEX).astype(np.int32)  # lookup seed on other side
    
    i  = np.concatenate((i, i, i))
    v1 = np.concatenate((voronoi.t[:, 1], voronoi.t[:, 2], voronoi.t[:, 0]))
    v2 = np.concatenate((voronoi.t[:, 2], voronoi.t[:, 0], voronoi.t[:, 1]))

    # Remove (i,j,v1,v2) index quadruplets that correspond to
    #   - border triangle edges (j == NO_INDEX)
    #   - triangle edges inside Laguerre cell (i == j)
    qidx = np.column_stack((i, j, v1, v2))
    qidx = qidx[np.logical_and(i != j, j != NO_INDEX)]

    i = qidx[:, 0]  # re-extract i, j, v1, v2
    j = qidx[:, 1]
    v1 = qidx[:, 2]
    v2 = qidx[:, 3]

    return i, j, v1, v2


def triangle_area(vertices, triangles):
    """
    @brief Computes the area of a mesh triangle
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] triangles an array with the three vertices indices of the triangle
    @details Works also when T is an array of triangles (then it returns
        the array of triangle areas). This is why the ellipsis (...)
        is used (here it means indexing/slicing through the last dimension)
    """
    v1 = triangles[..., 0]
    v2 = triangles[..., 1]
    v3 = triangles[..., 2]

    u = vertices[v2] - vertices[v1]
    v = vertices[v3] - vertices[v1]
    return np.abs(0.5 * (u[..., 0] * v[..., 1] - u[..., 1] * v[..., 0]))


def distance(vertices, v1, v2):
    """
    @brief Computes the length of a mesh edge
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] v1, v2 the mesh extremities indices.
    @details v1 and v2 can be also arrays (then returns the array of distances).
    """
    axis = v1.ndim if hasattr(v1, 'ndim') else 0
    return np.linalg.norm(vertices[v2] - vertices[v1], axis=axis)


def hessian(voronoi):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """
    NO_INDEX = -1  # Special value for invalid indices (edge on border)

    # Compute one entry per triangle half-edge (3*nt entries) with:
    # I is the seed associated with the triangle
    # J is the seed on the other side of the triangle's edge (NO_INDEX on border)
    # V1 and V2 are the two vertices of the triangle
    i = voronoi.tseed

    tadj = voronoi.tadj
    tadj = tadj[:, [1, 2, 0]] # <- permute columns to match std convention for triangulations:
                              #   different in geogram meshes because they also support n-sided polygons
                                      
    j = tadj.T.flatten().astype(np.int32)
    j = np.where(j != NO_INDEX, i[j], NO_INDEX).astype(np.int32)  # lookup seed on other side
    
    i  = np.concatenate((i, i, i))
    v1 = np.concatenate((voronoi.t[:, 1], voronoi.t[:, 2], voronoi.t[:, 0]))
    v2 = np.concatenate((voronoi.t[:, 2], voronoi.t[:, 0], voronoi.t[:, 1]))

    # Remove (i,j,v1,v2) index quadruplets that correspond to
    #   - border triangle edges (j == NO_INDEX)
    #   - triangle edges inside Laguerre cell (i == j)
    qidx = np.column_stack((i, j, v1, v2))
    qidx = qidx[np.logical_and(i != j, j != NO_INDEX)]

    i = qidx[:, 0]  # re-extract i, j, v1, v2
    j = qidx[:, 1]
    v1 = qidx[:, 2]
    v2 = qidx[:, 3]

    # Now we can compute the vector of coefficient (note: v1, v2, i, j are vectors)
    val = -distance(voronoi.q, v1, v2) / (2. * distance(voronoi.seeds, i, j))

    return i, j, val


def measures(voronoi):
    """
    @brief Computes the measures of the Laguerre cells
    @return the vector of Laguerre cells measures
    @details Uses the current Laguerre diagram (in self.Laguerre)
    """
    # See comments about XY,T,trgl_seed,nt in compute_Laguerre_diagram()
    measures = np.zeros(len(voronoi.seeds))
    np.add.at(measures, voronoi.tseed, triangle_area(voronoi.q, voronoi.t))
    return measures

As well as our Optimal Transport solver based on KMT's algorithm:

In [3]:
class Transport:
    def __init__(self, domain=None, use_direct_solver: bool = True, verbose: bool = False):
        """
        @brief Transport constructor
        @param[in] seeds seeds coordinates
        @param[in] domain Voronoi diagram domain
        @param[in] use_direct_solver: direct (if set) or iterative solver otherwise
        @param[in] verbose log Newton iterations if set
        """

        self.direct = use_direct_solver
        self.verbose = verbose

        self.domain = domain
        if self.domain is None:
            self.domain = geo.shape.quad()

        # Parameters for linear solver
        self.regularization = 0.0
        if self.direct:  # if using direct solver, one needs regulariz.
            self.regularization = 1e-6  # because matrix is singular ([1,1...1] in ker)


    def solve(self, seeds):
        dimension = seeds.shape[1]
        assert dimension == 2, 'seeds must be a (N, 2) array.'

        # Compute Laguerre diagram
        psi     = np.zeros(len(seeds), np.float64)
        voronoi = geo.Voronoi(seeds, psi, domain_vertices=self.domain[0], domain_simplices=self.domain[1])

        # Measure of whole domain, desired areas and minimum legal area (KMT #1)
        areas = measures(voronoi)
        assert np.min(areas) > 0., 'Voronoi diagram has an empty cell.'

        omega_measure = np.sum(areas)  # Measure of the whole domain
        nu_i = omega_measure / len(seeds)  # Desired area for each cell
        area_threshold = 0.5 * min(np.min(areas), nu_i)  # KMT criterion  #1

        threshold = nu_i * 0.001  # 0.1% of desired cell area
        current_error = np.inf

        i = 0
        while current_error > threshold and i < 100:
            H = self.hessian(seeds, voronoi, nu_i)  # Hessian of Kantorovich dual (sparse matrix)

            # rhs (minus gradient of Kantorovich dual) = desired areas - actual areas
            b = nu_i - measures(voronoi)
            if self.regularization != 0.0:
                b -= self.regularization * nu_i * psi

            g_norm = np.linalg.norm(b)  # norm of gradient at current step (KMT #2)
            p = self.solve_linear_system(H, b)  # solve for p in H*p=b
            alpha = 1.0  # Steplength
            psi += p  # Start with Newton step

            # Divide steplength by 2 until both KMT criteria are satisfied
            for ii in range(100):
                voronoi = geo.Voronoi(seeds, psi, domain_vertices=self.domain[0], domain_simplices=self.domain[1])

                # g (grad of Kantorovich dual) at substep = actual areas - desired areas
                g = measures(voronoi)
                smallest_area = np.min(g)  # for KMT criterion 1
                g -= nu_i

                # Check KMT criteria #1 (cell area) and #2 (gradient norm)
                g_norm_k = np.linalg.norm(g)
                kmt_1 = (smallest_area > area_threshold)  # criterion 1: cell area
                kmt_2 = (g_norm_k <= (1.0 - 0.5 * alpha) * g_norm)  # criterion 2: gradient norm

                if self.verbose:
                    print(f' KMT #1 (area): {kmt_1} {smallest_area}>{area_threshold}')
                    print(f' KMT #2 (grad): {kmt_2} {g_norm_k}<={(1.0 - 0.5 * alpha) * g_norm}')
                    
                if kmt_1 and kmt_2:
                    break

                alpha = alpha / 2.0
                psi -= alpha * p

            if ii == 100:
                print('Error: Did not converged!')

            current_error = np.linalg.norm(b, ord=np.inf)
            i += 1

        if i == 100:
            print('Error: Did not converged!')
        
        return voronoi

    def solve_linear_system(self, H, b):
        """
        @brief Solves a linear system
        @details Works in direct or iterative mode, with scipy and with OpenNL
        @param[in] H the matrix of the linear system
        @param[in] b the right hand side
        @return p such that H p = b
        """
        if self.direct:
            p = scipy.sparse.linalg.spsolve(H, b)
        else:
            linalg = scipy.sparse.linalg
            dim = (len(b), len(b))

            # A: operator:       y <- (H + diag)*x
            # M: preconditioner: y <- diag@{-1}*x
            self.iter = 0
            p, info = linalg.cg(
                A=linalg.LinearOperator(dim, matvec=lambda x: H @ x + H.diag * x),
                b=b,
                M=linalg.LinearOperator(dim, matvec=lambda x: x / H.diag),
                callback=print if self.verbose else None,
                atol=0.0,  # normally the default, but larger on older scipy ver.
                rtol=1e-3  # or tol=1e-3 instead of rtol, depends on scipy ver.
            )
            if info != 0:
                print(f'CG did not converge, info={info}', info)
        return p


    def hessian(self, seeds, voronoi, nu_i):
        """
        @brief Computes the matrix of the system to be solved at each Newton step
        @details Uses the current Laguerre diagram (in self.Laguerre). Works in
         scipy and in OpenNL mode. In the (scipy,iterative) combination, the
         diagonal of the matrix is stored separately in a dynamically created
         'diag' field of the returned scipy sparse matrix.
        @return the Hessian matrix of the Kantorovich dual
        """

        i, j, val = hessian(voronoi)

        n = len(seeds)
        diag = np.zeros(n, np.float64)  # Diagonal (initialized to zero)
        np.add.at(diag, i, -val)  # =minus sum extra-diagonal coefficients
        if self.regularization != 0.0:
            diag += self.regularization * nu_i

        H = scipy.sparse.csr_array((val, (i, j)), shape=(n, n))
        if self.direct:  # if using direct solver, inject diag coeffs into mtx
            s = np.arange(n, dtype=np.int32)
            H += scipy.sparse.csr_array((diag, (s, s)), shape=(n, n))
        else:
            H.diag = diag  # store diagonal separately if using iterative solver

        return H

Finally, we need some display utilities.

In [4]:
def plot_particles_sequence(voronois, labels):
    img_nb    = len(voronois)
    _, axes = plt.subplots( int( np.ceil(img_nb / 4) ), min( img_nb, 4 ) )
    for img in range(img_nb):
        current = voronois[img]
        
        ax = axes if img_nb == 1 else (axes[img] if img_nb <= 4 else axes[img // 4, img % 4])
            
        triangulation = tri.Triangulation(current.q[:, 0], current.q[:, 1], current.t)
        ax.tripcolor(triangulation, labels[current.tseed], shading='flat', cmap='tab20', antialiased=True, edgecolors='face', linewidth=0.5)
        ax.add_patch( matplotlib.patches.Rectangle((0, 0), 1, 1, color='black', fc = 'none',lw = 1))

        ax.set_aspect('equal')
        ax.margins(0.01)
        ax.axis('off')


def save_particle_video(voronois, labels, out_folder, figsize=(10, 10)):
    os.makedirs(os.getcwd() + os.sep + out_folder, exist_ok=True)

    img_nb = len(voronois)

    fig, ax = plt.subplots( 1, 1, figsize=figsize )
    ax.set_aspect('equal')
    ax.margins(0.01)
    ax.axis('off')

    trail_len = 1
    offsets = np.vstack([centroids(voronois[0])] * trail_len)
    colors = np.concatenate([labels] * trail_len)

    sc = ax.scatter(
        offsets[:, 0], offsets[:, 1],
        c=colors, s=12.5,
        cmap='tab20',
        edgecolors='face',
        vmin=0, vmax=19,
        linewidth=0.72
    )

    max_points = trail_len * len(voronois[0].seeds)
    for i in trange(img_nb):
        
        current = centroids(voronois[i])
        offsets = np.vstack((offsets, current))

        if len(offsets) > max_points:
            offsets = offsets[-max_points:]
            colors = colors[-max_points:]

        sc.set_offsets(offsets)
        sc.set_array(colors)

        # Save figure
        os.makedirs(os.getcwd() + '/out', exist_ok=True)
        fig.savefig(os.getcwd() + os.sep + out_folder + os.sep + '{0:04d}.png'.format(i))
        plt.close()


def save_particle_video_voronoi(voronois, labels, out_folder, figsize=(10, 10)):
    os.makedirs(os.getcwd() + os.sep + out_folder, exist_ok=True)

    img_nb = len(voronois)

    for i in trange(img_nb):
        fig, ax = plt.subplots( 1, 1, figsize=figsize )
        ax.set_aspect('equal')
        ax.margins(0.01)
        ax.axis('off')

        current = voronois[i]

        triangulation = tri.Triangulation(current.q[:, 0], current.q[:, 1], current.t)
        ax.tripcolor(triangulation, labels[current.tseed], cmap='tab20', shading='flat', antialiased=True, edgecolors='face', linewidth=0.72)
        ax.add_patch( matplotlib.patches.Rectangle((0, 0), 1, 1, color='black', fc = 'none',lw = 1))
        
        _, _, v1, v2 = facets(current)
        lines = np.c_[current.q[v1], current.q[v2]].reshape(len(v1), 2, 2)
        
        lc = LineCollection(lines, colors='black', linewidths=.2, alpha=1, antialiaseds=True)
        ax.add_collection(lc)

        barycenters = centroids(current)
        ax.scatter(barycenters[:, 0], barycenters[:, 1], c='black', s=.25)

        # Save figure
        os.makedirs(os.getcwd() + '/out', exist_ok=True)
        fig.savefig(os.getcwd() + os.sep + out_folder + os.sep + '{0:04d}.png'.format(i))
        plt.close()

We can now fully implement de Goes et al.'s numerical scheme. This is made through the definition of 3 discrete differential operators: the divergence, the gradient and the Laplacian.

In [5]:
def divergence(voronoi, field):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """
    i, j, v1, v2 = facets(voronoi)
    
    # Now we can compute the vector of coefficient (note: v1, v2, i, j are vectors)
    hij = distance(voronoi.q, v1, v2) / distance(voronoi.seeds, i, j)

    bij = (voronoi.q[v1] + voronoi.q[v2]) * .5
    qj  = voronoi.seeds[j, :]
    qi  = voronoi.seeds[i, :]
    val = hij[:, None] * (qj - bij)
    val_diag = hij[:, None] * (qi - bij)

    n = len(voronoi.seeds)
    diag = np.zeros((n, 2), np.float64)  # Diagonal (initialized to zero)
    np.add.at(diag, i, -val_diag)  # =minus sum extra-diagonal coefficients

    # D = np.full((n, n, 2), fill_value=0.)
    # D[i, j] = val
    # 
    # idx = np.arange(n)
    # D[idx, idx, :] = diag
    
    D0 = scipy.sparse.csr_array((val[:, 0], (i, j)), shape=(n, n)) + scipy.sparse.diags(diag[:, 0])
    D1 = scipy.sparse.csr_array((val[:, 1], (i, j)), shape=(n, n)) + scipy.sparse.diags(diag[:, 1])

    return D0 @ field[:, 0] + D1 @ field[:, 1]


def gradient(voronoi, field):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """
    i, j, v1, v2 = facets(voronoi)
    
    # Now we can compute the vector of coefficient (note: v1, v2, i, j are vectors)
    hij = distance(voronoi.q, v1, v2) / distance(voronoi.seeds, i, j)

    bij = (voronoi.q[v1] + voronoi.q[v2]) * .5
    qi  = voronoi.seeds[i, :]

    derivatives = hij[:, None] * (qi - bij) * (field[i] - field[j])[:, None]
    
    result = np.zeros((len(voronoi.seeds), 2), np.float64)
    np.add.at(result, i, derivatives)

    return result


def laplacian(voronoi, masses):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """
    i, j, v1, v2 = facets(voronoi)
    
    # Now we can compute the vector of coefficient (note: v1, v2, i, j are vectors)
    hij = distance(voronoi.q, v1, v2) / distance(voronoi.seeds, i, j)

    bij = (voronoi.q[v1] + voronoi.q[v2]) * .5
    qj  = voronoi.seeds[j, :]
    qi  = voronoi.seeds[i, :]
    val = hij[:, None] * (qj - bij)
    val_diag = hij[:, None] * (qi - bij)

    n    = len(voronoi.seeds)
    diag = np.zeros((n, 2), np.float64)  # Diagonal (initialized to zero)
    np.add.at(diag, i, -val_diag)    # =minus sum extra-diagonal coefficients
    
    D0 = scipy.sparse.csr_array((val[:, 0], (i, j)), shape=(n, n)) + scipy.sparse.diags(diag[:, 0])
    D1 = scipy.sparse.csr_array((val[:, 1], (i, j)), shape=(n, n)) + scipy.sparse.diags(diag[:, 1])
    
    # L = (D @ diag(m)^-1 @ G) = (D @ diag(m)^-1 @ -D.T) = -(D @ diag(m)^-1 @ D.T)
    m  = scipy.sparse.diags(1. / masses)
    G0 = -D0.T
    G1 = -D1.T
    L = (D0 @ m @ G0 + D1 @ m @ G1)
    
    return L

Finally, we can implement the stepping.

In [6]:
(domain_vertices, domain_triangles) = geo.shape.quad()
optimizer = Transport((domain_vertices, domain_triangles), verbose=False)

def power_particles_step(seeds, velocities, masses, tau=.001, g=9.81, mu=1e-6 ):
    voronoi = optimizer.solve(seeds)
    
    # Compute forces: F = - m G Z
    f = np.c_[np.zeros(len(masses)), -masses * g]
    L = laplacian(voronoi, masses)

    if mu > 0.:
        nu   = np.ones(len(seeds)) / len(seeds)
        A    = scipy.sparse.diags(nu) - mu * tau * L
        v0, _ = scipy.sparse.linalg.cg(
            A=A,
            b=nu * (velocities[:, 0] + tau * f[:, 0] / masses),
        )
        v1, _ = scipy.sparse.linalg.cg(
            A=A,
            b=nu * (velocities[:, 1] + tau * f[:, 1] / masses),
        )

        v = np.c_[v0, v1]
    else:
        v = velocities + (tau / masses[:, np.newaxis]) * f
    
    div = divergence(voronoi, v)

    # Note that L is negative semi-definite, so we solve -L p = -div instead of L p = div
    p, info = scipy.sparse.linalg.cg(
        A=-tau*L,
        b=-div,
    )
    if info != 0:
        print(f"CG did not converge, info={info}")
    
    v = v - (tau / masses[:, None]) * gradient(voronoi, p)

    # print(np.max(divergence(voronoi, v)))
    c = centroids(voronoi)
    s = c + tau * v
    return s, v, voronoi

## Taylor-Green Vortex

In [11]:
seeds = np.random.rand(10000, 2)
for i in range(4):
    seeds = centroids(optimizer.solve(seeds))

velocities = np.zeros_like(seeds)
masses     = np.ones(len(seeds))

# Taylor-Green
velocities = np.c_[
     np.cos(np.pi * .5 + 2. * np.pi * seeds[:, 0]) * np.sin(np.pi * .5 + 2. * np.pi * seeds[:, 1]),
    -np.sin(np.pi * .5 + 2. * np.pi * seeds[:, 0]) * np.cos(np.pi * .5 + 2. * np.pi * seeds[:, 1])
]

last_seeds      = seeds
last_velocities = velocities

animation       = []
for i in trange(200):
    last_seeds, last_velocities, voronoi = power_particles_step(last_seeds, last_velocities, masses, tau=.01, g=0., mu=0.)
    last_seeds = np.clip(last_seeds, 0., 1.)

    animation += [ voronoi ]

if True:
    bands = np.floor(seeds[:, 0] * 20).astype(int)   # 20 vertical stripes
    # save_particle_video_voronoi(animation, bands, 'out/power-particles-taylor-green')
    save_particle_video(animation, bands, 'out/power-particles-taylor-green')
else:
    plot_particles_sequence(animation[::2], np.arange(len(INITIAL_SEEDS)))

100%|██████████| 200/200 [00:23<00:00,  8.64it/s]


## Rayleigh–Taylor instability

In [8]:
seeds = np.random.rand(10000, 2)
for i in range(4):
    seeds = centroids(optimizer.solve(seeds))
    
velocities = np.zeros_like(seeds)
masses     = np.ones(len(seeds))

labels = (seeds[:, 1] - .5) > .1 * np.sin(seeds[:, 0] * 10.)
masses[labels] = 3.

labels = np.array(labels).astype(int)
labels[labels == 1] = 2

last_seeds = seeds
last_velocities = velocities

animation       = []
for i in trange(200):
    last_seeds, last_velocities, voronoi = power_particles_step(last_seeds, last_velocities, masses, tau=.01, g=9.81, mu=0.)
    last_seeds = np.clip(last_seeds, 0., 1.)
    
    animation += [ voronoi ]

if True:
    # save_particle_video_voronoi(animation, labels, 'out/power-particles-instability')
    save_particle_video(animation, labels, 'out/power-particles-instability')

else:
    plot_particles_sequence(animation[::2], np.arange(len(INITIAL_SEEDS)))

100%|██████████| 200/200 [00:22<00:00,  8.77it/s]
